# Phase C — SmolVLA on Kaggle

This notebook fine-tunes the official `lerobot/smolvla_base` checkpoint on the private, sanitized Phase B dataset. It does not upload the trained policy.

Before running:

1. Select **GPU T4 x2** and turn **Internet on** in Kaggle settings.
2. Add a Kaggle secret named `HF_TOKEN` with read access to `stevenzenith/hand_tracking_pv_carton_phase_b`.
3. Use **Save Version → Save & Run All** for the full run so `/kaggle/working` is retained.

Frozen first-run contract: 30 episodes / 6640 frames / 10 Hz; current front and side images, current 6-D state and stored task text; native 50-step SmolVLA training chunk; `n_action_steps=1`; frozen VLM/vision encoder; action-expert fine-tuning only.


In [ ]:
# Install the same LeRobot source revision used by the local Phase C run.
import importlib.metadata as metadata
import subprocess
import sys

LEROBOT_COMMIT = "da92db8fc0c935950a56b1ea61fa9b211ef3ac30"
assert sys.version_info >= (3, 12), (
    f"This pinned LeRobot revision requires Python >=3.12; Kaggle has {sys.version.split()[0]}"
)

package = f"lerobot[training,smolvla] @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        package,
        "huggingface_hub==1.19.0",
        "transformers==5.5.4",
    ],
    check=True,
)

for name in ["lerobot", "torch", "torchvision", "torchcodec", "transformers", "accelerate", "huggingface_hub"]:
    print(f"{name}={metadata.version(name)}")


In [ ]:
# Authenticate, verify the requested accelerator, and pin/download both Hub inputs.
import os
from pathlib import Path

SCRATCH_ROOT = Path("/kaggle/temp/phase_c_smolvla")
WORK_ROOT = Path("/kaggle/working/phase_c_smolvla")
DATASET_ROOT = SCRATCH_ROOT / "dataset"
MODEL_ROOT = SCRATCH_ROOT / "smolvla_base"
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(SCRATCH_ROOT / "hf_home")
os.environ["HF_DATASETS_CACHE"] = str(SCRATCH_ROOT / "hf_home" / "datasets")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download
import torch

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_memory_gib = [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2) for i in range(torch.cuda.device_count())]
print("GPUs:", list(zip(gpu_names, gpu_memory_gib, strict=True)))
print("CUDA:", torch.version.cuda, "native_bf16:", torch.cuda.is_bf16_supported())
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    "Select the Kaggle GPU T4 x2 accelerator before continuing"
)

DATASET_ID = "stevenzenith/hand_tracking_pv_carton_phase_b"
DATASET_REVISION = "1bb681ab58b5ca2cbdedb52dabf8e1f7a6052a6a"
BASE_MODEL_ID = "lerobot/smolvla_base"
BASE_MODEL_REVISION = "c83c3163b8ca9b7e67c509fffd9121e66cb96205"

dataset_info = api.dataset_info(DATASET_ID, revision=DATASET_REVISION)
model_info = api.model_info(BASE_MODEL_ID, revision=BASE_MODEL_REVISION)
assert dataset_info.private is True
assert dataset_info.sha == DATASET_REVISION
assert model_info.sha == BASE_MODEL_REVISION
print("dataset revision:", dataset_info.sha, "private:", dataset_info.private)
print("base model revision:", model_info.sha)

snapshot_download(
    repo_id=DATASET_ID, repo_type="dataset", revision=DATASET_REVISION,
    local_dir=DATASET_ROOT, token=hf_token,
)
snapshot_download(
    repo_id=BASE_MODEL_ID, revision=BASE_MODEL_REVISION,
    local_dir=MODEL_ROOT, token=hf_token,
)
del hf_token


In [ ]:
# Verify the exact dataset/model feature contract and decode representative AV1 frames.
import json
from lerobot.datasets.lerobot_dataset import LeRobotDataset

EXPECTED_TASK = (
    "Gently grasp and lift the 250 g paper carton, tighten the gripper if it slips, "
    "then return it to the table and release it."
)
dataset = LeRobotDataset(
    repo_id=DATASET_ID, root=DATASET_ROOT, revision=DATASET_REVISION,
    video_backend="torchcodec",
)
assert dataset.num_episodes == 30
assert dataset.num_frames == 6640
assert dataset.fps == 10
assert set(dataset.meta.camera_keys) == {"observation.images.front", "observation.images.side"}

sample = dataset[0]
assert tuple(sample["observation.images.front"].shape) == (3, 480, 640)
assert tuple(sample["observation.images.side"].shape) == (3, 480, 640)
assert tuple(sample["observation.state"].shape) == (6,)
assert tuple(sample["action"].shape) == (6,)
assert sample["task"] == EXPECTED_TASK

base_config = json.loads((MODEL_ROOT / "config.json").read_text())
assert base_config["type"] == "smolvla"
assert base_config["chunk_size"] == 50
assert base_config["load_vlm_weights"] is True
assert base_config["input_features"]["observation.state"]["shape"] == [6]
assert base_config["output_features"]["action"]["shape"] == [6]
assert {"observation.images.camera1", "observation.images.camera2"}.issubset(base_config["input_features"])

print("dataset:", dataset.num_episodes, "episodes /", dataset.num_frames, "frames /", dataset.fps, "Hz")
print("sample shapes:", {key: tuple(sample[key].shape) for key in ["observation.images.front", "observation.images.side", "observation.state", "action"]})
print("task:", sample["task"])
print("base chunk:", base_config["chunk_size"], "base cameras:", [key for key in base_config["input_features"] if key.startswith("observation.images.")])
del dataset, sample


In [ ]:
# Build the fixed two-GPU command and stream a persistent log without shell interpolation.
import datetime as dt
import json
import shutil
import subprocess

TRAIN_CLI = shutil.which("lerobot-train")
ACCELERATE = shutil.which("accelerate")
assert TRAIN_CLI and ACCELERATE
PER_GPU_BATCH = 2
RENAME_MAP = json.dumps({
    "observation.images.front": "observation.images.camera1",
    "observation.images.side": "observation.images.camera2",
}, separators=(",", ":"))

def build_command(*, steps: int, output_dir: Path, job_name: str, save_checkpoint: bool, save_freq: int, log_freq: int) -> list[str]:
    warmup_steps = 500 if steps >= 20_000 else 20
    decay_steps = 20_000 if steps >= 20_000 else 20
    train_args = [
        f"--policy.path={MODEL_ROOT}",
        f"--dataset.repo_id={DATASET_ID}",
        f"--dataset.root={DATASET_ROOT}",
        f"--dataset.revision={DATASET_REVISION}",
        "--dataset.video_backend=torchcodec",
        f"--rename_map={RENAME_MAP}",
        f"--batch_size={PER_GPU_BATCH}",
        "--num_workers=2",
        "--policy.chunk_size=50",
        "--policy.n_action_steps=1",
        "--policy.freeze_vision_encoder=true",
        "--policy.train_expert_only=true",
        "--policy.load_vlm_weights=true",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--policy.push_to_hub=false",
        f"--policy.scheduler_warmup_steps={warmup_steps}",
        f"--policy.scheduler_decay_steps={decay_steps}",
        f"--steps={steps}",
        "--eval_freq=0",
        f"--save_checkpoint={str(save_checkpoint).lower()}",
        f"--save_freq={save_freq}",
        f"--log_freq={log_freq}",
        f"--output_dir={output_dir}",
        f"--job_name={job_name}",
        "--seed=1000",
        "--wandb.enable=false",
    ]
    return [
        ACCELERATE, "launch", "--multi_gpu", "--num_processes=2",
        "--mixed_precision=no", "--main_process_port=29517",
        TRAIN_CLI, *train_args,
    ]

def run_and_log(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    child_env["TQDM_DISABLE"] = "1"
    print("command:", subprocess.list2cmdline(command))
    with log_path.open("w", buffering=1) as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=child_env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training exited with code {return_code}; inspect {log_path}")


In [ ]:
# Mandatory 20-update distributed smoke: model load, language path, AV1 decode, forward/backward and memory.
import re
import statistics

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
SMOKE_OUTPUT = WORK_ROOT / "outputs" / f"smoke_{stamp}"
SMOKE_LOG = WORK_ROOT / "logs" / f"smoke_{stamp}.log"
smoke_command = build_command(
    steps=20, output_dir=SMOKE_OUTPUT, job_name="smolvla_phase_c_smoke",
    save_checkpoint=False, save_freq=20, log_freq=1,
)
run_and_log(smoke_command, SMOKE_LOG)

smoke_text = SMOKE_LOG.read_text(errors="replace")
timings = [(float(update), float(data)) for update, data in re.findall(r"updt_s:([0-9.]+).*?data_s:([0-9.]+)", smoke_text)]
memory = [float(value) for value in re.findall(r"mem_gb:([0-9.]+)", smoke_text)]
assert timings, "Smoke completed without parseable timing metrics"
steady = timings[-min(5, len(timings)):]
seconds_per_step = statistics.mean(update + data for update, data in steady)
estimated_hours = seconds_per_step * 20_000 / 3600
print(f"smoke passed: {seconds_per_step:.3f} s/step steady, peak logged memory {max(memory):.2f} GiB")
print(f"projected 20k update time: {estimated_hours:.2f} h, excluding setup and checkpoints")
if estimated_hours > 10.5:
    print("WARNING: 20k may cross Kaggle's 12-hour limit; 5k checkpoints make a later resume possible.")


In [ ]:
# Full first baseline. This intentionally starts from smolvla_base, not from the smoke weights.
FULL_STEPS = 20_000
full_stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
FULL_OUTPUT = WORK_ROOT / "outputs" / f"smolvla_phase_c_20k_{full_stamp}"
FULL_LOG = WORK_ROOT / "logs" / f"smolvla_phase_c_20k_{full_stamp}.log"
full_command = build_command(
    steps=FULL_STEPS, output_dir=FULL_OUTPUT, job_name="smolvla_phase_c_20k",
    save_checkpoint=True, save_freq=5_000, log_freq=50,
)
run_and_log(full_command, FULL_LOG)

manifest = {
    "completed_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "lerobot_commit": LEROBOT_COMMIT,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "base_model_id": BASE_MODEL_ID,
    "base_model_revision": BASE_MODEL_REVISION,
    "gpus": gpu_names,
    "per_gpu_batch": PER_GPU_BATCH,
    "global_batch": PER_GPU_BATCH * len(gpu_names),
    "steps": FULL_STEPS,
    "chunk_size": 50,
    "n_action_steps": 1,
    "freeze_vision_encoder": True,
    "train_expert_only": True,
    "rename_map": json.loads(RENAME_MAP),
    "smoke_seconds_per_step": seconds_per_step,
    "smoke_projected_20k_hours": estimated_hours,
    "output_dir": str(FULL_OUTPUT),
    "log_path": str(FULL_LOG),
    "command": full_command,
}
manifest_path = WORK_ROOT / "smolvla_phase_c_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("manifest:", manifest_path)


In [ ]:
# Confirm retained checkpoints before saving the Kaggle notebook version.
checkpoint_models = sorted((FULL_OUTPUT / "checkpoints").glob("*/pretrained_model/model.safetensors"))
assert checkpoint_models, f"No checkpoints found under {FULL_OUTPUT}"
for model_path in checkpoint_models:
    print(model_path.relative_to(WORK_ROOT), f"{model_path.stat().st_size / 2**20:.1f} MiB")
print("full log:", FULL_LOG)
print("Now save this Kaggle notebook version so /kaggle/working/phase_c_smolvla is retained.")
